# Model Exploration

This notebook is the trial-and-error workspace for Part II modeling (see `CONVENTIONS.md`). The final, clean version — refactored into functions — moves to `model_pipeline/` once the approach is settled.

## Step 1: train/test split

**Is it necessary to split before imputing? Yes — and it is worth being explicit about why.** Any statistic used to fill missing values (a median, a category frequency) has to be *fit* on some data. If that data includes the test set, the test set has quietly influenced training, so the later evaluation is no longer a clean estimate of how the model performs on data it has never seen. The correct order is: split first, then fit every preprocessing step (imputer, encoder, and later the model itself) only on the training data, and only ever *apply* — never *fit* — those same fitted steps to the test data. This is why the split happens in this cell, before the `ColumnTransformer` in Step 2 is even built.

The four columns that will be imputed each have exactly 30 nulls in the full 1,000-row dataset (3.0% each, per `EDA_report.md`): `Weather`, `Traffic_Level`, `Time_of_Day`, and `Courier_Experience_yrs`. `Order_ID` is dropped as a plain identifier with no predictive value, and the target is separated out before the split.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/Food_Delivery_Times.csv")

X = df.drop(columns=["Order_ID", "Delivery_Time_min"])
y = df["Delivery_Time_min"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}, y_test: {y_test.shape}")

X_train: (800, 7), X_test: (200, 7)
y_train: (800,), y_test: (200,)


In [2]:
impute_cols = ["Weather", "Traffic_Level", "Time_of_Day", "Courier_Experience_yrs"]

null_breakdown = pd.DataFrame({
    "n_null_full": df[impute_cols].isnull().sum(),
    "pct_null_full": (df[impute_cols].isnull().mean() * 100).round(1),
    "n_null_train": X_train[impute_cols].isnull().sum(),
    "pct_null_train": (X_train[impute_cols].isnull().mean() * 100).round(1),
    "n_null_test": X_test[impute_cols].isnull().sum(),
    "pct_null_test": (X_test[impute_cols].isnull().mean() * 100).round(1),
})
null_breakdown

,n_null_full,pct_null_full,n_null_train,pct_null_train,n_null_test,pct_null_test
Weather,30,3.0,22,2.8,8,4.0
Traffic_Level,30,3.0,19,2.4,11,5.5
Time_of_Day,30,3.0,24,3.0,6,3.0
Courier_Experience_yrs,30,3.0,24,3.0,6,3.0


The 3.0% null rate holds exactly in the full dataset, but the random split does **not** reproduce that 3.0% identically in train and test — e.g. `Traffic_Level` is 2.4% null in train vs. 5.5% in test, and `Weather` is 2.8% vs. 4.0%. This is expected sampling variation from a single random 80/20 split, and it is itself part of why train and test have to stay separate: they are not identical copies of each other, so a statistic fit on one is not automatically valid for the other.

**A concrete illustration of the leakage risk**, using `Courier_Experience_yrs`: the median computed on the full 1,000-row dataset is **5.0**, identical to the median computed on the 800-row training split alone (**5.0**) — for this particular column and split, imputing "the wrong way" would happen to produce the same fill value. But the *mean* tells a different story: **4.579** on the full dataset vs. **4.607** on the training split alone — a small but real difference (≈ -0.03). The median coinciding here is a property of this specific dataset (coarse, integer-like values), not a guarantee; the mean shows the underlying mechanism plainly, and with a smaller dataset, a different split, or a different imputation statistic, the gap would not necessarily stay this small. This is exactly the leakage `X_train`/`X_test` avoids: every fitted value used later comes only from the 800 training rows.

## Step 2: preprocessing pipeline (`ColumnTransformer`)

This follows the strategy already decided in `EDA_report.md`: median imputation for the three numeric columns; `"Unknown"` imputation followed by encoding for the four categorical columns, split into one ordinal column (`Traffic_Level`) and three nominal columns (`Weather`, `Time_of_Day`, `Vehicle_Type`).

**On `Traffic_Level`'s `"Unknown"` position:** `Traffic_Level` is genuinely ordinal (`Low < Medium < High`), but `"Unknown"` is not really a point on that scale — it is missing information, not a fourth traffic level. It is placed *after* `High` (`Low=0, Medium=1, High=2, Unknown=3`) rather than before `Low` or in the middle, because the EDA found that rows with a missing `Traffic_Level` had a *higher* average delivery time (62.1 min) than the overall average (56.6 min) — closer in direction to `High` traffic than to `Low`. This is still a judgment call, not a fact derived from the missing values themselves (which are, by definition, unobserved); if this ordinal treatment turns out to hurt model performance, treating `Traffic_Level` as a fourth nominal (one-hot) category instead would be a reasonable alternative to test.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

numeric_cols = ["Distance_km", "Preparation_Time_min", "Courier_Experience_yrs"]
ordinal_cols = ["Traffic_Level"]
nominal_cols = ["Weather", "Time_of_Day", "Vehicle_Type"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OrdinalEncoder(categories=[["Low", "Medium", "High", "Unknown"]])),
])

nominal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("ordinal", ordinal_pipeline, ordinal_cols),
    ("nominal", nominal_pipeline, nominal_cols),
])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('ordinal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``f

That output is `scikit-learn`'s interactive HTML diagram (its default rendering for estimators in a Jupyter notebook, not just printed text) — every box is clickable/expandable, showing the exact nesting of the `ColumnTransformer`, its three named branches, and each `Pipeline`'s steps with their fitted parameters. It is a genuinely useful sanity check on its own: a quick visual scan confirms each column list is routed to the right imputer/encoder pair before anything is actually fit, catching a mis-wired transformer far faster than reading the constructor code line by line.

## Step 3: quick validation

`fit_transform` is called only on `X_train`, so every fitted value (the medians, the one-hot categories seen) comes exclusively from training data. `X_test` is only ever `transform`-ed, never fit.

In [4]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"X_train_transformed: {X_train_transformed.shape}")
print(f"X_test_transformed: {X_test_transformed.shape}")
print(f"Nulls remaining in train: {np.isnan(X_train_transformed).sum()}")
print(f"Nulls remaining in test: {np.isnan(X_test_transformed).sum()}")

X_train_transformed: (800, 18)
X_test_transformed: (200, 18)
Nulls remaining in train: 0
Nulls remaining in test: 0


In [5]:
preprocessor.get_feature_names_out()

array(['numeric__Distance_km', 'numeric__Preparation_Time_min',
       'numeric__Courier_Experience_yrs', 'ordinal__Traffic_Level',
       'nominal__Weather_Clear', 'nominal__Weather_Foggy',
       'nominal__Weather_Rainy', 'nominal__Weather_Snowy',
       'nominal__Weather_Unknown', 'nominal__Weather_Windy',
       'nominal__Time_of_Day_Afternoon', 'nominal__Time_of_Day_Evening',
       'nominal__Time_of_Day_Morning', 'nominal__Time_of_Day_Night',
       'nominal__Time_of_Day_Unknown', 'nominal__Vehicle_Type_Bike',
       'nominal__Vehicle_Type_Car', 'nominal__Vehicle_Type_Scooter'],
      dtype=object)

18 output columns from the 7 input columns: 3 numeric (imputed, unchanged in count) + 1 ordinal (`Traffic_Level`, encoded to a single numeric column) + 14 one-hot columns from the 3 nominal features (6 `Weather` levels including `Unknown`, 5 `Time_of_Day` levels including `Unknown`, 3 `Vehicle_Type` levels — no `Unknown` needed there, since it has no nulls). No model has been trained yet — that is the next step.

## Step 4: missing-value check and post-transformation summary statistics

The transformed arrays are wrapped into DataFrames using `preprocessor.get_feature_names_out()` for column labels — this is only for inspection here, not part of the pipeline itself. A per-column missing-value count (not just the single total already shown in Step 3) confirms exactly which columns are clean, and `describe()` gives a short statistical summary of every transformed variable, for both the training and test sets.

In [6]:
feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)

missing_check = pd.DataFrame({
    "n_missing_train": X_train_df.isnull().sum(),
    "n_missing_test": X_test_df.isnull().sum(),
})

assert missing_check["n_missing_train"].sum() == 0
assert missing_check["n_missing_test"].sum() == 0
print("Confirmed: 0 missing values in every column, both train and test.")
missing_check

Confirmed: 0 missing values in every column, both train and test.


,n_missing_train,n_missing_test
numeric__Distance_km,0,0
numeric__Preparation_Time_min,0,0
numeric__Courier_Experience_yrs,0,0
ordinal__Traffic_Level,0,0
nominal__Weather_Clear,0,0
nominal__Weather_Foggy,0,0
nominal__Weather_Rainy,0,0
nominal__Weather_Snowy,0,0
nominal__Weather_Unknown,0,0
nominal__Weather_Windy,0,0


In [7]:
X_train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
numeric__Distance_km,800.0,10.10055,5.725900,0.59,5.13,10.22,14.9425,19.99
numeric__Preparation_Time_min,800.0,17.08375,7.227519,5.00,11.00,17.00,23.0000,29.00
numeric__Courier_Experience_yrs,800.0,4.61875,2.902832,0.00,2.00,5.00,7.0000,9.00
ordinal__Traffic_Level,800.0,0.85750,0.814262,0.00,0.00,1.00,1.0000,3.00
nominal__Weather_Clear,800.0,0.46875,0.499335,0.00,0.00,0.00,1.0000,1.00
nominal__Weather_Foggy,800.0,0.10250,0.303494,0.00,0.00,0.00,0.0000,1.00
nominal__Weather_Rainy,800.0,0.20500,0.403954,0.00,0.00,0.00,0.0000,1.00
nominal__Weather_Snowy,800.0,0.09625,0.295118,0.00,0.00,0.00,0.0000,1.00
nominal__Weather_Unknown,800.0,0.02750,0.163637,0.00,0.00,0.00,0.0000,1.00
nominal__Weather_Windy,800.0,0.10000,0.300188,0.00,0.00,0.00,0.0000,1.00


The numeric columns look consistent with the EDA (e.g. `Distance_km` mean ≈ 10.10, `Preparation_Time_min` mean ≈ 17.08), now with no nulls left to fill. `ordinal__Traffic_Level` ranges from 0 to 3 as expected (`Low`-`Unknown`), with a mean of 0.86 — pulled toward the low end, since `Low` and `Medium` are the two largest categories. Every one-hot column is a proportion between 0 and 1, and each mean matches its category's share of the training data — e.g. `nominal__Weather_Clear` ≈ 0.47 lines up with `Clear` being roughly half of all rows, and the small `_Unknown` means (`Weather` ≈ 0.03, `Time_of_Day` ≈ 0.03) line up with the ~3% null rate found in the EDA, now carried through as their own category instead of a missing value.

## Step 5: does imputation preserve the original statistical properties?

This compares each column's statistics on `X_train` *before* imputation (computed only on the non-null rows, via `.dropna()`) against the same column *after* imputation — both on the training split, so the comparison is fair and like-for-like. The goal is to check directly whether filling missing values changed the data's behavior, rather than assuming it did not.

In [8]:
numeric_comparison = []
for c in numeric_cols:
    before = X_train[c].dropna()
    after = X_train_df[f"numeric__{c}"]
    numeric_comparison.append({
        "column": c,
        "n_missing": len(after) - len(before),
        "mean_before": before.mean(), "mean_after": after.mean(),
        "median_before": before.median(), "median_after": after.median(),
        "std_before": before.std(), "std_after": after.std(),
        "std_change_pct": 100 * (after.std() - before.std()) / before.std(),
    })

pd.DataFrame(numeric_comparison).round(3)

,column,n_missing,mean_before,mean_after,median_before,median_after,std_before,std_after,std_change_pct
0,Distance_km,0,10.101,10.101,10.22,10.22,5.726,5.726,0.000
1,Preparation_Time_min,0,17.084,17.084,17.00,17.00,7.228,7.228,0.000
2,Courier_Experience_yrs,24,4.607,4.619,5.00,5.00,2.947,2.903,-1.487


In [9]:
for c, prefix in [("Weather", "Weather"), ("Time_of_Day", "Time_of_Day"), ("Vehicle_Type", "Vehicle_Type")]:
    before_pct = (X_train[c].value_counts(normalize=True, dropna=True) * 100).round(2)
    after_cols = [col for col in X_train_df.columns if col.startswith(f"nominal__{prefix}_")]
    after_pct = (X_train_df[after_cols].mean() * 100).round(2)
    after_pct.index = [i.replace(f"nominal__{prefix}_", "") for i in after_pct.index]
    print(f"\n{c}")
    print(pd.DataFrame({"before_pct": before_pct, "after_pct": after_pct}).fillna(0.0))


Weather
         before_pct  after_pct
Clear         48.20      46.88
Foggy         10.54      10.25
Rainy         21.08      20.50
Snowy          9.90       9.62
Unknown        0.00       2.75
Windy         10.28      10.00

Time_of_Day
           before_pct  after_pct
Afternoon       28.87       28.0
Evening         30.41       29.5
Morning         30.93       30.0
Night            9.79        9.5
Unknown          0.00        3.0

Vehicle_Type
         before_pct  after_pct
Bike          49.62      49.62
Car           20.38      20.38
Scooter       30.00      30.00


In [10]:
before_pct = (X_train["Traffic_Level"].value_counts(normalize=True, dropna=True) * 100).round(2)
after_pct = (
    X_train_df["ordinal__Traffic_Level"]
    .map({0: "Low", 1: "Medium", 2: "High", 3: "Unknown"})
    .value_counts(normalize=True) * 100
).round(2)

pd.DataFrame({"before_pct": before_pct, "after_pct": after_pct}).fillna(0.0)

,before_pct,after_pct
High,20.36,19.88
Low,39.82,38.88
Medium,39.82,38.88
Unknown,0.00,2.38


**The honest answer is: mostly, but not perfectly — imputation has small, real, measurable effects, not zero effect.**

- `Distance_km` and `Preparation_Time_min` have no missing values at all, so there is nothing to compare — before and after are identical by construction.
- `Courier_Experience_yrs` (24 imputed rows in `X_train`, ~3%): the **mean is nearly unchanged** (4.607 → 4.619, a +0.012 shift) and the **median is exactly unchanged** (5.0 → 5.0, since the imputed values *are* the median). But the **standard deviation shrinks by ~1.5%** (2.947 → 2.903). This is the expected mechanical effect of median imputation: replacing missing values with a single constant removes variation that would otherwise have been there, slightly compressing the spread. At a 3% missing rate the effect is small, but it is not zero, and it would grow if the missing rate were higher.
- `Weather`, `Time_of_Day`, and `Traffic_Level` (each ~3% missing): introducing `"Unknown"` as its own category necessarily **dilutes every other category's share by roughly the missing rate** — e.g. `Weather`'s `Clear` share drops from 48.20% (among non-null rows) to 46.88% (once `Unknown` claims its own 2.75%); `Traffic_Level`'s `Low` and `Medium` each drop by about 1 point. The *relative ordering* of categories is preserved, but their *exact* proportions are not — by design, since `"Unknown"` was never part of the original category distribution and is now competing for share with the real ones.
- `Vehicle_Type` has no missing values, so it is unchanged, as expected.

So imputation does not leave the data's behavior completely untouched: it modestly compresses the numeric column's spread and dilutes categorical proportions in proportion to how much was missing. Given the missing rate here is only ~3% per affected column, these shifts are small — but they are a real, if minor, trade-off of the chosen strategy, not something to claim is fully preserved.